### Step 0: Cohort Data Ingestion & Tensor Assembly

Load and aggregate preprocessed 3D epoch arrays, target labels, and metadata across all 102 benchmark subjects ($X \in \mathbb{R}^{N_{\text{total\_trials}} \times 64 \times 641}$, $y \in \{0, 1\}^{N_{\text{total\_trials}}}$).

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import mne

# Add source directory to Python path
# sys.path.append(os.path.abspath('../srcs'))
from misc import load_and_parse_eeg

# Suppress verbose MNE logs
mne.set_log_level('WARNING')

# 1. Define Dataset Path & Benchmark Cohort (102 Clean Subjects)
BASE_DATA_PATH = "mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0"
ANOMALY_EXCLUDED_SUBJECTS = [38, 88, 89, 92, 100, 104, 106]
CLEAN_SUBJECTS = [s for s in range(1, 110) if s not in ANOMALY_EXCLUDED_SUBJECTS]
RUN_IDS = [4, 8, 12]  # Unilateral Motor Imagery: Left vs. Right Fist


In [3]:


print(f"Cohort Configuration:")
print(f"  Total Subjects Requested : {len(CLEAN_SUBJECTS)}")
print(f"  Excluded Anomaly Subjects: {ANOMALY_EXCLUDED_SUBJECTS}")
print(f"  Target Run IDs           : {RUN_IDS}\n")



Cohort Configuration:
  Total Subjects Requested : 102
  Excluded Anomaly Subjects: [38, 88, 89, 92, 100, 104, 106]
  Target Run IDs           : [4, 8, 12]



In [4]:


# 2. Ingest, Filter, and Epoch Cohort Data into 3D Tensor
X, y, df_metadata = load_and_parse_eeg(
    subject_ids=CLEAN_SUBJECTS,
    run_ids=RUN_IDS,
    base_path=BASE_DATA_PATH,
    tmin=0.0,
    tmax=4.0,
    include_rest=False
)

Processing Subjects:   0%|          | 0/102 [00:00<?, ?it/s]

Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R08.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R12.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R04.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R08.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R12.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 mon

In [8]:
# 3. Cohort-Wide Integrity & Dimensional Verification
unique_subjects = np.unique(df_metadata['subject_id'])
unique_subjects


array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  39,  40,
        41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,
        54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,
        67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,  79,
        80,  81,  82,  83,  84,  85,  86,  87,  90,  91,  93,  94,  95,
        96,  97,  98,  99, 101, 102, 103, 105, 107, 108, 109])

In [9]:

class_counts = np.bincount(y)
class_counts

array([2304, 2267])

In [ ]:

print("--- Cohort Data Ingestion Integrity Summary ---")
print(f"Epoch Tensor (X) Shape    : {X.shape} (dtype: {X.dtype})")
print(f"Target Vector (y) Shape   : {y.shape} (dtype: {y.dtype})")
print(f"Metadata Records Shape    : {df_metadata.shape}")
print(f"Unique Subjects Loaded    : {len(unique_subjects)} / 102 (Target: 102)")
print(f"Class Balance (0 vs 1)    : Class 0 = {class_counts[0]}, Class 1 = {class_counts[1]} (Ratio: {class_counts[0]/class_counts[1]:.2f})")
print(f"Signal Amplitude (uV/V)   : Mean = {X.mean():.4e}, Std = {X.std():.4e}, Min = {X.min():.4e}, Max = {X.max():.4e}")

--- Cohort Data Ingestion Integrity Summary ---
Epoch Tensor (X) Shape    : (4571, 64, 641) (dtype: float64)
Target Vector (y) Shape   : (4571,) (dtype: int64)
Metadata Records Shape    : (4571, 5)
Unique Subjects Loaded    : 102 / 102 (Target: 102)
Class Balance (0 vs 1)    : Class 0 = 2304, Class 1 = 2267 (Ratio: 1.02)
Signal Amplitude (uV/V)   : Mean = -5.9437e-25, Std = 1.4706e-05, Min = -1.0648e-03, Max = 1.0420e-03


In [11]:

# Assert cohort integrity constraints
assert len(unique_subjects) == 102, f"Expected 102 unique subjects, got {len(unique_subjects)}"
assert X.ndim == 3 and X.shape[1] == 64 and X.shape[2] == 641, f"Unexpected tensor shape: {X.shape}"
assert set(np.unique(y)) == {0, 1}, f"Unexpected target classes: {np.unique(y)}"
print("\n[PASSED] Step 0 Cohort Data Ingestion and Tensor Assembly successfully verified.")


[PASSED] Step 0 Cohort Data Ingestion and Tensor Assembly successfully verified.
